# Start with the Input

In [22]:
import json
from itertools import groupby

from common.llm.typedllm import OllamaAiMessage

%load_ext autoreload
%autoreload 1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import os

while not "data" in os.listdir(os.getcwd()):
    os.chdir("..")

project = "data" + os.sep + "ecommerce-system"
requirements = project + os.sep + "requirements.txt"
scenarios = os.listdir(project + os.sep + "scenarios")
architectures = os.listdir(project + os.sep + "architectures")

In [24]:
from common.llm_access import create_prompt
from atam.datatypes_old import ScenarioMapping
%aimport atam.datatypes

def read_file(filename):
    with open(filename) as f:
        return json.load(f)


scenario_data = [read_file(project + os.sep + "scenarios" + os.sep + f) for f in scenarios]
scenario_data = [scenario for scenarios in scenario_data for scenario in scenarios]
architecture_data = [read_file(project + os.sep + "architectures" + os.sep + f) for f in architectures]

# these are not needed anymore; keep the jupyter context clean
del project, scenarios, architectures

In [25]:
# Save / load data
import pickle
from ipywidgets import Button

btn_import = Button(description='import data')
btn_export = Button(description='export data')
display(btn_import, btn_export)

data = ["scenario_mapping", "sensitivity_score"]


def path(filename):
    return os.path.join("checkpoints", filename)


def import_data(_):
    for export_var in data:
        print(export_var)
        if os.path.exists(path(export_var)):
            print("found")
            with open(path(export_var), "rb") as f:
                globals()[export_var] = pickle.load(f)
                print("imported")


def export_data(_):
    for export_var in data:
        print(export_var)
        if export_var in globals():
            print("found")
            with open(path(export_var), "wb") as f:
                pickle.dump(globals()[export_var], f)
                print("exported")


btn_import.on_click(import_data)
btn_export.on_click(export_data)
# no need to keep them in context
del btn_import, btn_export, import_data, export_data

Button(description='import data', style=ButtonStyle())

Button(description='export data', style=ButtonStyle())

In [26]:
from common.llm_access import create_typed_llm

llm = create_typed_llm(model="katjana-architectures-test", host="localhost:8000")
llm.create_model('katjana-architectures-test',
                 "llama3.2:3b",
                 "You are an assistant tasked to aid in taking design decisions for a software system.")

In [27]:
from ipywidgets import IntProgress
from common.llm_access import AiMessage


def generate_scenario_mapping(scenarios: list, architecture: any, debug) -> tuple[dict[str, AiMessage[list[ScenarioMapping]]], dict[str, Exception]]:
    success: dict[str, OllamaAiMessage[list[ScenarioMapping]]] = {}
    errors: dict[str, Exception] = {}
    f = IntProgress(min=0, max=len(scenarios)) # instantiate the bar
    display(f)
    for scenario in scenarios:
        try:
            success[scenario["name"]] = llm.generate(list[ScenarioMapping],
                                                  create_prompt(file="data/prompts/atam-1-scenario-mapping.md",
                                                                architecture_data=repr(architecture), scenario=repr(scenario)),
                                                  debug=debug, use_example_for_formatting=True)
        except Exception as err:
            errors[scenario["name"]] = err
        f.value += 1
    return success, errors

In [28]:
import datetime


def debug_printer(filename: str):
    def debug_print[T](information: T):
        with open("logs" + os.sep + filename, "a", encoding="utf8") as log:
            timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            for line in information.__repr__().strip().split('\n'):
                log.write(f"{timestamp} - {line}\n")
        return information

    try:
        os.remove("logs" + os.sep + filename)
    except FileNotFoundError:
        pass
    #with open("logs" + os.sep + filename, "x") as _:
    #    pass
    return debug_print

In [29]:
#############################################################
# STEP 1: SCENARIO MAPPING
#############################################################
# for each scenario, find their high impact components
#############################################################
architecture = f"```json\n{architecture_data[0]}\n```"
debug = debug_printer("scenario-mapping.txt")
scenario_mapping, errors = generate_scenario_mapping(scenario_data, architecture, debug)
display(f"{len(errors)} Errors")

IntProgress(value=0, max=22)

'1 Errors'

In [30]:
# retry errors
temp_scenario_mapping, errors = generate_scenario_mapping([scenario for scenario in scenario_data if scenario["name"] in errors], architecture, debug)
scenario_mapping = scenario_mapping | temp_scenario_mapping
del temp_scenario_mapping
display(f"{len(errors)} Errors")

IntProgress(value=0, max=2)

'1 Errors'

In [31]:
for scenario in scenario_mapping:
    for mapping in scenario_mapping[scenario].response:
        mapping.scenario = scenario

In [32]:
from itertools import groupby

all_mappings = sum([value.response for value in scenario_mapping.values()], []) # put all in one list
mappings_per_component = sorted(all_mappings, key=lambda x: x.component) # sort it
mappings_per_component = {k: list(g) for k, g in groupby(mappings_per_component, key=lambda x: x.component)} # group by component

In [33]:
#############################################################
# STEP 2: Sensitivity Identification
#############################################################
# Find sensitive points that strongly affect a quality attribute
#############################################################

# S(component) = ( max(impact)/3 + sum(impact)/max_impact ) / 2
max_per_component = {}
sum_per_component = {}
max_sum = 0

for mapping in all_mappings:
    component = mapping.component
    # max(impact[component])
    max_per_component[component] = max(max_per_component.get(component, 0), mapping.impact.numeric_value())
    # sum(impact[component])
    sum_per_component[component] = sum_per_component.get(component, 0) + mapping.impact.numeric_value()
    # max(sum(impact[component]) for all components)
    max_sum = max(max_sum, sum_per_component[component])

sensitivity_score = {k: (max_per_component[k] / 3 + sum_per_component[k] / max_sum) / 2 for k in
         max_per_component}
sensitivity_score = sorted(sensitivity_score.items(), key=lambda x: x[1], reverse=True)

del max_per_component, sum_per_component, max_sum

In [34]:
#############################################################
# STEP 3: Find trade offs
#############################################################
# Find components where the quality attributes are fighting each other
#############################################################

tradeoffs = {}
for component in mappings_per_component.keys():
    mappings = mappings_per_component[component]
    if any(mapping1.positive != mapping2.positive for mapping1 in mappings for mapping2 in mappings):
        tradeoffs[component] = {
            "positive": [mapping for mapping in mappings if mapping.positive],
            "negative": [mapping for mapping in mappings if not mapping.positive]
        }


In [35]:
errors

{'Scaling Under Peak Load': 1 validation error for list[ScenarioMapping]
   Invalid JSON: expected `,` or `}` at line 1 column 31 [type=json_invalid, input_value='[{"component":"Load Bala...oses it (required) */}]', input_type=str]
     For further information visit https://errors.pydantic.dev/2.12/v/json_invalid}

In [36]:
[scenario for scenario in scenario_data if scenario["name"] in errors]

[{'name': 'Scaling Under Peak Load',
  'Attribute': 'Scalability',
  'Environment': 'The system operates in a production environment with variable user traffic, including occasional peak loads triggered by external events.',
  'Stimulus': 'A sudden surge in concurrent user activity, such as 10x the average traffic within a short time window (e.g., during a flash sale or unplanned event).',
  'Response': 'The system dynamically scales resources to handle the increased load without degrading response times beyond 3 seconds for 95% of requests, ensuring system stability and maintaining core functionality.'},
 {'name': 'Scaling Under Peak Load',
  'Attribute': 'Scalability',
  'Environment': 'The system operates in a production environment with variable user traffic, including occasional peak loads triggered by external events.',
  'Stimulus': 'A sudden surge in concurrent user activity, such as 10x the average traffic within a short time window (e.g., during a flash sale or unplanned even